# B07 — Type 1 end-to-end `/predict` eval (vs round-1 gold)

Replays the **Type 1** questions from a round submission log
(`outputs/exact_eval_round1_Prompt2Win.json`) through our live `/predict`
pipeline, scores them the competition way, and exports a results JSON.

**Scoring (Type 1):** `sample_score = 0.5·P1 + 0.5·P2`
- **P1** = answer correct (100/0); `Unknown` and `Uncertain` treated as equal.
- **P2** = F1 over the 0-based `premises_used` index sets (gold vs predicted).

The round log supplies the gold (`expected.answer`, `expected.premises_used`),
so re-running measures how our current system would score on that exact set.

In [240]:
import json, re, time, asyncio, statistics
from pathlib import Path
from collections import Counter

import httpx

API_BASE    = "https://api.iamphuckhang.dev"
# Translator A/B (no server restart): None = server default (EXACT_TYPE1_TRANSLATOR),
# "single_pass" = whole-theory translation, "decompose" = per-premise pipeline.
TRANSLATOR  = "single_pass"
PREDICT_URL = f"{API_BASE}/predict" + (f"?translator={TRANSLATOR}" if TRANSLATOR else "")
# Keep concurrency modest: one GPU serves all requests, so high concurrency just
# queues and inflates per-request latency (false ReadTimeouts). TIMEOUT > the
# server-side request deadline (55s) so the server's own answer wins the race.
CONCURRENCY = 6
TIMEOUT     = 60.0

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "outputs").exists():
    ROOT = ROOT.parent
INPUT_JSON  = ROOT / "outputs/exact_eval_round1_Prompt2Win.json"
_tag = f"_{TRANSLATOR}" if TRANSLATOR else ""
OUTPUT_JSON = ROOT / f"outputs/B07_type1_eval{_tag}_{time.strftime('%Y%m%d_%H%M%S')}.json"
assert INPUT_JSON.exists(), f"round log not found: {INPUT_JSON}"
print("input   :", INPUT_JSON)
print("output  :", OUTPUT_JSON)
print("endpoint:", PREDICT_URL)

input   : /home/phuckhang/MyWorkspace/Exact2026/outputs/exact_eval_round1_Prompt2Win.json
output  : /home/phuckhang/MyWorkspace/Exact2026/outputs/B07_type1_eval_single_pass_20260621_095147.json
endpoint: https://api.iamphuckhang.dev/predict?translator=single_pass


## Load Type 1 questions + gold from the round log

In [241]:
_OPTION_LINE = re.compile(r"^\s*([A-E])[.)]\s+(.+)$", re.MULTILINE)

raw  = json.load(open(INPUT_JSON))
logs = raw.get("logs", [])

type1_samples = []
for entry in logs:
    if entry.get("type") != "type1":
        continue
    rp  = entry.get("request_payload", {}) or {}
    exp = entry.get("expected", {}) or {}
    query = rp.get("query") or rp.get("question") or ""
    type1_samples.append({
        "query_id": rp.get("query_id") or entry.get("query_id"),
        "type": "type1",
        "query": query,
        "premises": rp.get("premises"),
        "options": rp.get("options"),
        "_gold": exp.get("answer"),
        "_gold_premises": list(exp.get("premises_used") or []),
        "_is_mcq": bool(_OPTION_LINE.search(query)),
    })

all_samples = type1_samples
n_mcq = sum(s["_is_mcq"] for s in type1_samples)
print(f"Type 1 loaded: {len(type1_samples)}  (MCQ {n_mcq}, polar/YNU {len(type1_samples) - n_mcq})")
print("\nExample:")
ex = type1_samples[0]
print(json.dumps({k: ex[k] for k in ("query_id", "type", "query", "premises", "options")},
                 indent=2, ensure_ascii=False)[:600])
print("gold:", ex["_gold"], "| gold premises_used:", ex["_gold_premises"])

Type 1 loaded: 25  (MCQ 13, polar/YNU 12)

Example:
{
  "query_id": "T1_0021",
  "type": "type1",
  "query": "Based on the museum conservation rules, which conclusion is logically supported?\nA. The Amber Amulet must be displayed in a climate-controlled case\nB. The Amber Amulet cannot be placed on public display\nC. The Amber Amulet needs pest treatment before storage\nD. The Amber Amulet lacks a provenance certificate",
  "premises": [
    "If an artifact has a humidity-control log and no pest-damage report, then it is storage-ready.",
    "If an artifact is storage-ready and has a provenance certificate, then it is eligible for exhibition.",
gold: A | gold premises_used: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Send to `/predict`

In [242]:
def to_payload(sample: dict) -> dict:
    """Convert a sample to the unified /predict request body."""
    body: dict = {
        "query_id": sample["query_id"],
        "type": sample["type"],
        "query": sample["query"],
    }
    if sample["premises"]:
        body["premises"] = sample["premises"]
    if sample["options"]:
        body["options"] = sample["options"]
    return body


async def call(client, sem, sample):
    async with sem:
        t0 = time.perf_counter()
        err, body = None, None
        try:
            r = await client.post(PREDICT_URL, json=to_payload(sample), timeout=TIMEOUT)
            r.raise_for_status()
            body = r.json()
        except Exception as e:
            err = repr(e)
    return {
        **sample,
        "_response": body,
        "_latency": time.perf_counter() - t0,
        "_error": err,
    }


async def run_eval(samples):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(s):
            nonlocal done
            res = await call(client, sem, s)
            done += 1
            if done % 5 == 0 or done == len(samples):
                print(f"  {done}/{len(samples)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(s) for s in samples))


print(f"Sending {len(all_samples)} requests (concurrency={CONCURRENCY})...")
t0 = time.perf_counter()
results = await run_eval(all_samples)
wall = time.perf_counter() - t0

errors  = [r for r in results if r["_error"]]
success = [r for r in results if not r["_error"]]
print(f"\nSuccess : {len(success)}/{len(results)}")
print(f"Errors  : {len(errors)}")
print(f"Wall    : {wall:.1f}s")

Sending 25 requests (concurrency=6)...
  25/25
Success : 25/25
Errors  : 0
Wall    : 98.9s


## Score each sample (P1 answer + P2 premise-F1)

In [243]:
_UNC = {"uncertain", "unknown"}

def _norm_answer(a) -> str:
    s = str(a).strip().lower()
    return "uncertain" if s in _UNC else s

def _premise_f1(gold: list[int], pred: list[int]) -> float:
    g, p = set(gold), set(pred)
    if not g and not p:
        return 100.0
    if not g or not p:
        return 0.0
    inter = len(g & p)
    return 100.0 * 2 * inter / (len(g) + len(p)) if inter else 0.0

scored = []
for r in results:
    resp = r["_response"][0] if r["_response"] else None
    pred_ans  = resp.get("answer") if resp else None
    pred_prem = resp.get("premises_used") if resp else None
    gold_ans, gold_prem = r["_gold"], r["_gold_premises"]

    answer_ok = resp is not None and _norm_answer(pred_ans) == _norm_answer(gold_ans)
    p1 = 100.0 if answer_ok else 0.0
    p2 = _premise_f1(gold_prem, pred_prem or [])
    scored.append({
        "query_id": r["query_id"],
        "is_mcq": r["_is_mcq"],
        "gold_answer": gold_ans,
        "pred_answer": pred_ans,
        "answer_ok": answer_ok,
        "gold_premises_used": gold_prem,
        "pred_premises_used": pred_prem,
        "p1_score": p1,
        "p2_score": round(p2, 2),
        "sample_score": round(0.5 * p1 + 0.5 * p2, 2),
        "explanation": resp.get("explanation") if resp else None,
        "fol": resp.get("fol") if resp else None,
        "latency_s": round(r["_latency"], 1),
        "error": r["_error"],
    })

print(f"{'sample':12s} {'kind':4s} {'gold':9s} {'pred':9s} {'P1':>4s} {'P2':>6s} {'score':>6s} {'time':>7s}")
print("-" * 70)
for x in scored:
    flag = "✓" if x["answer_ok"] else "✗"
    print(f"{x['query_id']:12s} {'MCQ' if x['is_mcq'] else 'YNU':4s} "
          f"{str(x['gold_answer']):9s} {str(x['pred_answer']):9s} "
          f"{x['p1_score']:>4.0f} {x['p2_score']:>6.1f} {x['sample_score']:>6.1f} "
          f"{x['latency_s']:>6.1f}s {flag}")
    if x["error"]:
        print(f"   !! {x['error']}")

# --- per-request latency stats ---
lat = sorted(x["latency_s"] for x in scored)
if lat:
    pct = lambda q: lat[min(len(lat) - 1, int(q * len(lat)))]
    print(f"\nlatency/request (s): mean={statistics.mean(lat):.1f}  p50={pct(0.5):.1f}  "
          f"p90={pct(0.9):.1f}  min={lat[0]:.1f}  max={lat[-1]:.1f}")
    slow = sorted(scored, key=lambda x: -x["latency_s"])[:3]
    print("slowest 3:", ", ".join(f"{x['query_id']}={x['latency_s']:.1f}s" for x in slow))

sample       kind gold      pred        P1     P2  score    time
----------------------------------------------------------------------
T1_0021      MCQ  A         A          100  100.0  100.0   31.5s ✓
T1_0031      MCQ  B         B          100   82.3   91.2   29.5s ✓
T1_0025      MCQ  C         C          100  100.0  100.0   29.5s ✓
T1_0027      MCQ  D         C            0    0.0    0.0   43.7s ✗
T1_0035      MCQ  D         D          100  100.0  100.0   31.6s ✓
T1_0039      MCQ  C         C          100  100.0  100.0   27.4s ✓
T1_0023      MCQ  A         Uncertain    0    0.0    0.0   20.4s ✗
T1_0033      MCQ  C         C          100  100.0  100.0   23.7s ✓
T1_0046      MCQ  A         A          100  100.0  100.0   21.0s ✓
T1_0013      MCQ  D         D          100  100.0  100.0   21.0s ✓
T1_0015      MCQ  B         B          100  100.0  100.0   18.1s ✓
T1_0007      MCQ  B         Uncertain    0    0.0    0.0   10.9s ✗
T1_0041      MCQ  A         A          100  100.0  100.0   1

## Summary + export results JSON

In [244]:
def _avg(xs):
    return round(sum(xs) / len(xs), 2) if xs else 0.0

def _block(rows):
    return {
        "n": len(rows),
        "answer_acc": _avg([100.0 if x["answer_ok"] else 0.0 for x in rows]),
        "p1_avg": _avg([x["p1_score"] for x in rows]),
        "p2_avg": _avg([x["p2_score"] for x in rows]),
        "sample_score_avg": _avg([x["sample_score"] for x in rows]),
        "p2_perfect": sum(1 for x in rows if x["p2_score"] == 100.0),
    }

mcq = [x for x in scored if x["is_mcq"]]
ynu = [x for x in scored if not x["is_mcq"]]
summary = {
    "overall": _block(scored),
    "mcq": _block(mcq),
    "ynu": _block(ynu),
    "errors": sum(1 for x in scored if x["error"]),
    "latency_mean_s": _avg([x["latency_s"] for x in scored]),
}

payload = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "endpoint": PREDICT_URL,
    "input_log": INPUT_JSON.name,
    "summary": summary,
    "results": scored,
}
OUTPUT_JSON.write_text(json.dumps(payload, indent=2, ensure_ascii=False))

print("=== Type 1 summary (this run) ===")
for k in ("overall", "mcq", "ynu"):
    b = summary[k]
    print(f"  {k:8s} n={b['n']:<3d} answer_acc={b['answer_acc']:5.1f}%  "
          f"P1={b['p1_avg']:5.1f}  P2={b['p2_avg']:5.1f}  "
          f"sample={b['sample_score_avg']:5.1f}  (P2=100 in {b['p2_perfect']}/{b['n']})")
print(f"  errors={summary['errors']}  latency_mean={summary['latency_mean_s']}s")
print(f"\nwrote → {OUTPUT_JSON}")

=== Type 1 summary (this run) ===
  overall  n=25  answer_acc= 88.0%  P1= 88.0  P2= 78.3  sample= 83.1  (P2=100 in 14/25)
  mcq      n=13  answer_acc= 76.9%  P1= 76.9  P2= 75.6  sample= 76.2  (P2=100 in 9/13)
  ynu      n=12  answer_acc=100.0%  P1=100.0  P2= 81.2  sample= 90.6  (P2=100 in 5/12)
  errors=0  latency_mean=21.44s

wrote → /home/phuckhang/MyWorkspace/Exact2026/outputs/B07_type1_eval_single_pass_20260621_095147.json


In [245]:
# --- Diagnostic table: where does each type1 sample stop? -------------------
# Separates real logical uncertainty (Z3_TRUE_UNCERTAIN) from parser/verifier/
# mode gaps so we know what to fix next.
hdr = f"{'sample':14s} {'used':5s} {'verif':5s} {'supp':5s} {'mode':16s} {'opt_fol':7s} {'cause'}"
print(hdr)
print("-" * len(hdr))
from collections import Counter as _C
cause_counts = _C()
for r in [x for x in success if x["type"] == "type1" and x["_response"]]:
    rd = r["_response"][0].get("routing_diagnostics") or {}
    qs = rd.get("query_spec") or {}
    n_opt_fol = sum(1 for c in qs.get("option_claims", []) if c.get("fol"))
    cause = rd.get("uncertainty_cause")
    cause_counts[cause] += 1
    print(f"{r['query_id']:14s} "
          f"{str(rd.get('solver_used')):5s} "
          f"{str(rd.get('premise_bundle_verified')):5s} "
          f"{str(qs.get('supported')):5s} "
          f"{str(qs.get('solver_mode')):16s} "
          f"{n_opt_fol:<7d} "
          f"{cause if cause is not None else '— (solved)'}")

print("\n=== uncertainty_cause distribution ===")
for c, n in cause_counts.most_common():
    print(f"  {str(c) if c is not None else '— (solved)':40s}: {n}")

# Premise warnings actually seen (non-blocking; solver still ran)
warn = _C()
for r in [x for x in success if x["type"] == "type1" and x["_response"]]:
    for w in (r["_response"][0].get("routing_diagnostics") or {}).get("premise_warnings", []):
        warn[w.split(":")[0]] += 1
if warn:
    print("\n=== premise warnings (non-blocking) ===")
    for w, n in warn.most_common():
        print(f"  {w:40s}: {n}")

sample         used  verif supp  mode             opt_fol cause
---------------------------------------------------------------
T1_0021        True  None  None  None             0       — (solved)
T1_0031        True  None  None  None             0       — (solved)
T1_0025        True  None  None  None             0       — (solved)
T1_0027        True  None  None  None             0       — (solved)
T1_0035        True  None  None  None             0       — (solved)
T1_0039        True  None  None  None             0       — (solved)
T1_0023        True  None  None  None             0       — (solved)
T1_0033        True  None  None  None             0       — (solved)
T1_0046        True  None  None  None             0       — (solved)
T1_0013        True  None  None  None             0       — (solved)
T1_0015        True  None  None  None             0       — (solved)
T1_0007        True  None  None  None             0       — (solved)
T1_0041        True  None  None  None       

## Notes
- Eval set = every `type1` entry in `INPUT_JSON` (the round log). Gold answer + gold `premises_used` come from each entry's `expected` block.
- `sample_score = 0.5·P1 + 0.5·P2`; P2 = F1 over 0-based premise-index sets. `Unknown`/`Uncertain` are scored as equal answers.
- Results JSON written to `OUTPUT_JSON` (timestamped under `outputs/`): per-sample `gold/pred answer`, `premises_used`, P1/P2/sample, `explanation`, `fol`, latency, error, plus an aggregate `summary`.
- Compare a new run's `summary.overall.sample_score_avg` against the round log's `summary.p1_score`/`p2_score` to measure improvement.
- Inspect one: `next(r for r in results if r['query_id'] == 'T1_0021')['_response']`.

## FOL translations (premises / question / options)

`/predict` returns the full response, so the FOL is carried inside each result's
`routing_diagnostics` — no extra API calls:

- **premises → FOL**: `routing_diagnostics.parsed_premises[]` (`original_text` → `fol`)
- **translated question**: `routing_diagnostics.query_spec.main_claim_text` → `main_claim_fol`
- **options → FOL**: `routing_diagnostics.query_spec.option_claims[]` (`label, role, claim_text` → `fol`)

In [246]:
# FOL comes from the /predict response in `success`. Two schemas:
#   decompose    -> routing_diagnostics.parsed_premises / query_spec
#   single_pass  -> routing_diagnostics.premise_fol / claim_fol / option_fol
t1_fol = [r for r in success if r["type"] == "type1" and r["premises"]]

def _has_fol(rd):
    return bool(rd.get("parsed_premises") or rd.get("premise_fol"))

have_fol = sum(1 for r in t1_fol if _has_fol(r["_response"][0].get("routing_diagnostics") or {}))
print(f"Type 1 samples: {len(t1_fol)}  |  carrying FOL detail: {have_fol}")

Type 1 samples: 25  |  carrying FOL detail: 25


In [247]:
for r in t1_fol:
    resp = r["_response"][0]
    rd = resp.get("routing_diagnostics") or {}
    print("=" * 88)
    print(f"[{r['query_id']}]  pred={resp.get('answer')!r}  gold={r['_gold']!r}  "
          f"({'MCQ' if r['_is_mcq'] else 'polar/YNU'})  stage={rd.get('stage')}")

    if rd.get("stage") == "single_pass_translation":
        # --- single-pass: original NL alongside the translated FOL ---
        print("\n  ORIGINAL NL PREMISES:")
        for i, p in enumerate(r.get("premises") or [], 1):
            print(f"    {i}. {p}")
        print("\n  PREMISES → FOL:")
        for f in rd.get("premise_fol", []):
            print(f"    ⇒ {f}")
        print("\n  ORIGINAL QUESTION:")
        for line in (r["query"] or "").splitlines():
            print(f"    {line}")
        print(f"\n  CLAIM → FOL:  {rd.get('claim_fol')}")
        if rd.get("option_fol"):
            print("  OPTIONS → FOL:")
            for lbl, f in rd["option_fol"].items():
                print(f"    {lbl}. ⇒ {f}")
        if rd.get("vote_distribution"):
            print(f"  votes: {rd['vote_distribution']}")
        if rd.get("refine_log"):
            print(f"  refine_log: {rd['refine_log']}")
        if rd.get("translation_issues"):
            print(f"  issues: {rd['translation_issues']}")
    else:
        # --- decompose: parsed_premises + query_spec (original_text included) ---
        qs = rd.get("query_spec") or {}
        print("\n  PREMISES → FOL")
        for p in rd.get("parsed_premises", []):
            print(f"    • {p['original_text']}")
            print(f"        ⇒ {p['fol']}")
        if rd.get("premise_bundle_verified") is False:
            print(f"    [verification issues: {rd.get('premise_verification_issues')}]")
        print("\n  QUESTION → FOL")
        print(f"    text : {qs.get('main_claim_text')}")
        print(f"    fol  : {qs.get('main_claim_fol')}")
        print(f"    mode : {qs.get('question_format')}/{qs.get('solver_mode')}  "
              f"supported={qs.get('supported')}  negate={qs.get('negate_claim')}")
        oc = qs.get("option_claims") or []
        if oc:
            print("\n  OPTIONS → FOL")
            for c in oc:
                text = c.get("claim_text") or c.get("normalized_text")
                print(f"    {c['label']}. [{c.get('role')}] {text}")
                print(f"        ⇒ {c.get('fol')}")
        if qs.get("issues"):
            print(f"\n  [query issues: {qs['issues']}]")
    print()

[T1_0021]  pred='A'  gold='A'  (MCQ)  stage=single_pass_translation

  ORIGINAL NL PREMISES:
    1. If an artifact has a humidity-control log and no pest-damage report, then it is storage-ready.
    2. If an artifact is storage-ready and has a provenance certificate, then it is eligible for exhibition.
    3. If an artifact is eligible for exhibition and has curator approval, then it can be placed on public display.
    4. If an artifact is fragile, then it requires low-light protection.
    5. If an artifact requires low-light protection and can be placed on public display, then it must be displayed in a climate-controlled case.
    6. The Amber Amulet has a humidity-control log.
    7. The Amber Amulet has no pest-damage report.
    8. The Amber Amulet has a provenance certificate.
    9. The Amber Amulet has curator approval.
    10. The Amber Amulet is fragile.

  PREMISES → FOL:
    ⇒ forall x: (HasHumidityControlLog(x) & ~HasPestDamageReport(x)) -> IsStorageReady(x)
    ⇒ forall 